# Regression tables for supplementary materials

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

import glob
import re
from funcs_support import get_params
dir_list = get_params()

In [97]:


for proc in ['mortality','gdppc','payroll']:
    print('\n\n')
    
    proc_names = {'gdppc':'GDP per capita growth',
                  'mortality':'Mortality rate',
                  'payroll':'Annual payroll'}
    
    fns = glob.glob(f'{dir_list['regs']}{proc}/*.dta')
    fns = [fn for fn in fns if 'ALLYEAR' not in fn]
    #fn = fns[0]
    def read_regdta(fn):
        df = pd.read_stata(fn)
        keep_rows = (df.parm.str.startswith('ta') | df.parm.str.startswith('c.temp') | df.parm.str.startswith('tem') | df.parm.str.startswith('pr') | df.parm.str.startswith('days'))
        df = df.loc[keep_rows,:]
        fn_splits = pd.DataFrame([re.split(r'\_',fn) for fn in fns])
        id_col = np.where(['GMFD' in fn_splits.iloc[:,x].values for x in range(fn_splits.shape[1])])[0][0]
        df['idstr'] = re.sub(r'nClim\-Grid','nClimGrid',re.sub('MERRA2','MERRA-2',re.sub(r'\-025','',re.split(r'\_',fn)[id_col])))
        return df
    
    
    
    df = pd.concat([read_regdta(fn) for fn in fns])
    
    tab = df.pivot(index="idstr", columns="parm", values=["estimate", "stderr", "p"])

    params = df['parm'].unique()
    # --- Anpassen -------------------------------------------------------------
    parm_dict = {'temp':r'$T$',
                 'c.temp#c.temp':r'$T^2$',
                 'precip':r'$P$',
                 'precip2':r'$P^2$',
                 **{f'tavg_poly_{x}':r'$T^'+str(x)+'$' for x in np.arange(1,5)},
                 'days90plus':r'# days > 90F',
                 'prec_dry':r'$P$'}
    
    header = {k:parm_dict[k] for k in params}
    
    oomags = np.floor(np.log10(np.abs(df.groupby('parm')['estimate'].mean())))
    scale = {k:10**v for k,v in oomags.items()}
    
    prod_order = ['ERA5','GMFD','JRA-3Q','MERRA-2','UDel','PRISM','Daymet','PRISM-Schlenker','nClimGrid','NLDAS']
    prods_tmp = [prod for prod in prod_order if prod in df['idstr'].unique()
                ]
    
    digits = 2
    # --------------------------------------------------------------------------
    
    tab = df.pivot(index="idstr", columns="parm", values=["estimate", "stderr", "p"])
    tab = tab.reindex(prod_order)
    
    def stars(p):
        return "^{***}" if p < .01 else "^{**}" if p < .05 else "^{*}" if p < .1 else ""
    
    def fmt_coef(prod, par):
        b = tab.loc[prod, ("estimate", par)] / scale[par]
        p = tab.loc[prod, ("p", par)]
        return f"${b:.{digits}f}{stars(p)}$"
    
    def fmt_se(prod, par):
        se = tab.loc[prod, ("stderr", par)] / scale[par]
        return f"$({se:.{digits}f})$"
    
    def exp10(x):                                  # 1e-3 -> "10^{-3}"
        import math
        return f"10^{{{int(round(math.log10(x)))}}}"
    
    # Spaltenköpfe mit Skalierungsfaktor
    head_cells = [f"{header[p]} ($\\times {exp10(scale[p])}$)" for p in params]
    
    lines = []
    lines.append(r"\begin{table}[htbp]\centering")
    lines.append(r"\caption{"+proc_names[proc]+" coefficients by data product}")
    lines.append(r"\label{tab:robustness_products}")
    lines.append(r"\begin{tabular}{l" + "c" * len(params) + "}")
    lines.append(r"\toprule")
    lines.append("Historical product & " + " & ".join(head_cells) + r" \\")
    lines.append(r"\midrule")
    for prod in prods_tmp:
        lines.append(prod + " & " + " & ".join(fmt_coef(prod, p) for p in params) + r" \\")
        lines.append(" & " + " & ".join(fmt_se(prod, p) for p in params) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\par\smallskip\footnotesize Standard errors in parentheses. "
                 r"$^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$.")
    lines.append(r"\end{table}")
    
    latex = "\n".join(lines)
    print(latex)

    # --------- CSV FOR EXCEL --> WORD --------- 
    tab_out = tab.dropna()
    tab_out = tab_out.reset_index().melt(id_vars=[('idstr', '')])
    tab_out.columns = ['Historical product','metric','varble','value']
    
    tab_out = tab_out.set_index(['Historical product','metric']).sort_index().pivot(columns='varble')
    
    prodlist = np.unique(tab_out.index.get_level_values('Historical product'))
    for prod in prodlist:
        for p in tab_out.columns.get_level_values('varble'):
            tab_out.loc[(prod,'output'),('value',p)] = fmt_coef(prod, p)
            tab_out.loc[(prod,'se'),('value',p)] = fmt_se(prod,p)
    
    tab_out = pd.concat([tab_out.sort_index().loc[[(prod,'output') for prod in prodlist],:],
               tab_out.sort_index().loc[[(prod,'se') for prod in prodlist],:]]).sort_index()
    tab_out.columns = tab_out.columns.droplevel()
    tab_out.columns = [parm_dict[v] for v in tab_out.columns]

    # Save as csv for excel --> word pipeline
    csv_out = f'{dir_list['aux']}regcoeffs_{proc}.csv'
    tab_out.to_csv(csv_out)
    print(csv_out+' saved!')




\begin{table}[htbp]\centering
\caption{Mortality rate coefficients by data product}
\label{tab:robustness_products}
\begin{tabular}{lcccc}
\toprule
Historical product & $T^1$ ($\times 10^{-3}$) & $T^2$ ($\times 10^{-3}$) & $T^3$ ($\times 10^{-6}$) & $T^4$ ($\times 10^{-6}$) \\
\midrule
ERA5 & $-4.54$ & $-1.14^{**}$ & $-38.46$ & $2.85^{***}$ \\
 & $(9.15)$ & $(0.51)$ & $(36.88)$ & $(0.97)$ \\
GMFD & $-7.73$ & $-1.91^{***}$ & $66.70^{*}$ & $0.01$ \\
 & $(10.95)$ & $(0.63)$ & $(34.09)$ & $(0.79)$ \\
JRA-3Q & $-7.43$ & $-1.52^{***}$ & $-14.56$ & $2.25^{***}$ \\
 & $(9.07)$ & $(0.49)$ & $(31.30)$ & $(0.83)$ \\
MERRA-2 & $-16.31^{*}$ & $-0.88^{**}$ & $-2.39$ & $1.46^{**}$ \\
 & $(8.73)$ & $(0.44)$ & $(31.96)$ & $(0.73)$ \\
\bottomrule
\end{tabular}
\par\smallskip\footnotesize Standard errors in parentheses. $^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$.
\end{table}
/glade/u/home/schwarzwald/projects/impact_uncertainty/aux_data/regcoeffs_mortality.csv saved!



\begin{table}[htbp]\centering
\

/glade/derecho/scratch/schwarzwald/tmp/ipykernel_8646/496590851.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$-4.54$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tab_out.loc[(prod,'output'),('value',p)] = fmt_coef(prod, p)
/glade/derecho/scratch/schwarzwald/tmp/ipykernel_8646/496590851.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$-1.14^{**}$' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tab_out.loc[(prod,'output'),('value',p)] = fmt_coef(prod, p)
/glade/derecho/scratch/schwarzwald/tmp/ipykernel_8646/496590851.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$-38.46$' has dtype incompatible with float64, please explicitly cast 

In [96]:
tab_out

# days > 90F            $P$
Historical product metric                              
Daymet             output  $-5.78^{***}$    $-2.74^{*}$
                   se           $(1.24)$       $(1.47)$
ERA5               output        $-1.87$         $2.11$
                   se           $(1.15)$       $(1.62)$
GMFD               output        $-1.52$        $-1.41$
                   se           $(1.22)$       $(1.37)$
JRA-3Q             output  $-4.44^{***}$   $-4.11^{**}$
                   se           $(1.18)$       $(1.80)$
MERRA-2            output  $-4.36^{***}$        $-1.93$
                   se           $(1.08)$       $(1.56)$
NLDAS              output  $-6.72^{***}$  $-6.25^{***}$
                   se           $(1.15)$       $(1.48)$
PRISM              output  $-4.11^{***}$   $-3.07^{**}$
                   se           $(1.18)$       $(1.46)$
PRISM-Schlenker    output  $-5.38^{***}$    $-2.63^{*}$
                   se           $(1.21)$       $(1.48)$
nClimGrid          output   $-2.99^{**}$        $-1.54$
                   se           $(1.22)$       $(1.50)$